In [1]:
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
%matplotlib inline

from networkx.algorithms import community
from networkx.algorithms.community import greedy_modularity_communities
from operator import itemgetter
import csv

In [3]:
cd /gpfs/group/smr48/default/io/

/gpfs/group/smr48/default/io


# Prepping Edges and Attributes

In [ ]:
edges_all = pd.DataFrame()
nodes_all = pd.DataFrame()


In [ ]:
country_list = ['bangladesh','catalonia', 'china', 'cuba', 'ecuador', 'egypt', 'honduras', 'indonesia', 'iran',
       'russia', 'serbia', 'saudi_arabia', 'spain', 'thailand', 'turkey', 'uae', 'uae_egypt', 'venezuela']


for country in country_list:
    edges = pd.read_csv('{}_coordination.csv'.format(country))
    edges['source_country'] = country
    edges = edges.rename(columns={"country": "target_country",
                                  "userid": "Source", "id":"Target"})
    edges.Source = edges.Source.apply(str)
    edges.Target = edges.Target.apply(str)
    edges = edges[edges['Source'] != edges['Target']] #filter all edges between source and target where they are same individual
    edges['target_country'] = edges['target_country'].str.replace('uae_egy','uae_egypt')
    edges = edges.loc[edges['target_country'] != country] #filter to outside connections only
    sources = edges.drop(['tweetid', 'accountDate', 'account_lang',
                                  'tweet_lang', 'text', 'tweet_time', 'tweetClient',
                                  'reply_user_id', 'quote_tweet_id', 'is_retweet',
                                  'retweet_user_id', 'hashtags', 'variable',
                                  'Target', 'target_country'], axis = 1).copy()
    targets = edges.drop(['tweetid', 'accountDate', 'account_lang',
                                  'tweet_lang', 'text', 'tweet_time', 'tweetClient',
                                  'reply_user_id', 'quote_tweet_id', 'is_retweet',
                                  'retweet_user_id', 'hashtags', 'variable',
                                  'Source', 'source_country'], axis = 1).copy()
    targets = targets.applymap(str)
    targets['type'] = 'target'
    sources = sources.applymap(str)
    sources['type'] = 'source' 
    sources = sources.rename(columns={'Source' : 'node', 'source_country': 'country'})
    targets = targets.rename(columns={'Target' : 'node', 'target_country': 'country'})
    nodes_updated = pd.concat([sources, targets])
    nodes_updated = nodes_updated.drop_duplicates()
    nodes_updated = nodes_updated.applymap(str)
    nodes_all = nodes_all.append(nodes_updated)
    nodes_all = nodes_all.drop_duplicates()
    print('Total Nodes after {}:{}'.format(country, str(len(nodes_all))))
    
    edges_updated = edges[['Source', 'Target', 'target_country', 'source_country', 'tweet_time']].copy()
    edges_updated.to_csv('{}_edgelist.csv'.format(country), index=False)
#     edges_updated = edges_updated.applymap(str)
    edges_updated_weight = edges_updated.groupby(["Source", "Target", "target_country", "source_country"]).size().reset_index(name="Weight")
    edges_updated_weight.columns = edges_updated_weight.columns.str.strip()
    edges_updated_weight = edges_updated_weight.sort_values(by=['Weight'], ascending=False)
    edges_updated_weight.to_csv('{}_edgelistWeighted.csv'.format(country), index=False)
#    edges_updated_weight
    edges_all = edges_all.append(edges_updated_weight)
    edges_all = edges_all.drop_duplicates()
    print('Total Weighted Edges: {} w/ Country: {}'.format(len(edges_all),country))

vc = nodes_all.node.value_counts() # count if value occurs more than once (source and a target)
vc = vc[vc>1]
ids = vc.index.tolist()
df = pd.DataFrame(ids,columns=['node'])
df['connection'] = 'S&T'  # create new value to signify both source and target
# df

print('Nodes_All: {}'.format(str(len(nodes_all))))
new_nodes = pd.merge(nodes_all, df, on=['node'], how='left')
print('Post-Merge w/ type S&T: {}'.format(str(len(new_nodes)))) #length should remain the same
# new_nodes


#Need to remove duplicates where subject is both a source and target which may have inflated node size
new_nodes.connection.fillna(new_nodes.type, inplace=True) #fill all NA values with either source or target from type column
del new_nodes['type']
new_nodes.columns = 'node country type'.split()
new_nodes = new_nodes.drop_duplicates() # remove duplicates 
print('Nodes_All: {}'.format(str(len(new_nodes))))

new_nodes.to_csv('allNodes.csv', index=False) #use for retriving all records and frequency tables
edges_all.to_csv('allEdges_Weighted.csv', index=False)


    
#     for country in country_list: 
#     records = pd.read_csv('{}_coordination.csv'.format(country))
#     print('{}: {}'.format(country, str(len(records))))
#     records = records[records.userid != 'userid']
#     print('{}: {}'.format(country, str(len(records))))
    

# Reading In Edges and Nodes | Building Graph

In [4]:
G = nx.Graph() # Initialize a Graph object


### 1a) Adding By Individual Country

In [ ]:
country_list = ['china', 'cuba', 'ecuador', 'egypt', 'honduras', 'indonesia', 'iran',
       'russia', 'saudi_arabia', 'serbia', 'turkey', 'uae', 'uae_egypt', 'venezuela']

for country in country_list:

    with open('{}_nodeslist.csv'.format(country), 'r', encoding="utf8") as nodecsv:
        nodereader = csv.reader(nodecsv)
        nodes = [n for n in nodereader][1:]

    # Get a list of just the node names (the first item in each row)
    node_names = [n[0] for n in nodes]
    node_names = list(map(str, node_names))

    # Read in the edgelist file
    with open('{}_edgelistWeighted.csv'.format(country), 'r', encoding='utf8') as edgecsv:
        edgereader = csv.reader(edgecsv)
        edges_full = [tuple(e) for e in edgereader][1:]

    source_country = {}
    target_country = {}
    weight = {}

    edges = [e[0:3] for e in edges_full]
    # edges = list(map(str, edges))

    # Print the number of nodes and edges in our two lists
    print('Nodes: {}'.format(len(node_names)))
    print('Edges: {}'.format(len(edges)))
    

    G.add_nodes_from(node_names) # Add nodes to the Graph
    G.add_weighted_edges_from(edges) # Add edges to the Graph
    print(nx.info(G)) # Print information about the Graph

    country_dict = {} 

    for node in nodes: # Loop through the list, one row at a time
        country_dict[node[0]] = node[1]

    nx.set_node_attributes(G, country_dict, 'country')



### 1B) Adding allEdges & allNodes

In [24]:
df = pd.read_csv('allEdges_Weighted.csv')

cols = df.columns.tolist()
cols = cols[:2] + cols[-1:] + cols[2:4]
df = df[cols]
df.to_csv('allEdges_Weighted.csv',index=False)
# df.to_csv('all')

df.sum()

In [29]:
df.sum()

Source            2670905189468980044427264003162243121100463249...
Target            4999718411399269998542218981306792846974254837...
Weight                                                        56008
target_country    saudi_arabiaindonesiaindonesiasaudi_arabiaindo...
source_country    chinachinachinachinachinachinachinachinachinac...
dtype: object

In [25]:
with open('allNodes.csv', 'r', encoding="utf8") as nodecsv:
    nodereader = csv.reader(nodecsv)
    nodes = [n for n in nodereader][1:]

# Get a list of just the node names (the first item in each row)
node_names = [n[0] for n in nodes]
node_names = list(map(str, node_names))

# Read in the edgelist file
with open('allEdges_Weighted.csv', 'r', encoding='utf8') as edgecsv:
    edgereader = csv.reader(edgecsv)
    edges_full = [tuple(e) for e in edgereader][1:]

source_country = {}
target_country = {}
weight = {}

edges = [e[0:3] for e in edges_full]
# edges = list(map(str, edges))

# Print the number of nodes and edges in our two lists
print('Nodes: {}'.format(len(node_names)))
print('Edges: {}'.format(len(edges)))
print('If Edges > Graph Edges Info...Bidirectional Edges Exist')

G.add_nodes_from(node_names) # Add nodes to the Graph
G.add_weighted_edges_from(edges) # Add edges to the Graph
print(nx.info(G)) # Print information about the Graph

country_dict = {} 

for node in nodes: # Loop through the list, one row at a time
    country_dict[node[0]] = node[1]

nx.set_node_attributes(G, country_dict, 'country')


Nodes: 3578
Edges: 6233
If Edges > Graph Edges Info...Bidirectional Edges Exist
Name: 
Type: Graph
Number of nodes: 3578
Number of edges: 5864
Average degree:   3.2778


In [28]:
# for n in G.nodes(): # Loop through every node, in our data "n" will be the name of the person
#     print(n, G.nodes[n]['country'])
    
density = nx.density(G)
print("Network density:", density)


# If your Graph has more than one component, this will return False:
print(nx.is_connected(G))

# Next, use nx.connected_components to get the list of components,
# then use the max() command to find the largest one:
components = nx.connected_components(G)
largest_component = max(components, key=len)

# Create a "subgraph" of just the largest component
# Then calculate the diameter of the subgraph, just like you did with density.
#

subgraph = G.subgraph(largest_component)
diameter = nx.diameter(subgraph)
print("Network diameter of largest component:", diameter)

triadic_closure = nx.transitivity(G)
print("Triadic closure:", triadic_closure)

degree_dict = dict(G.degree(G.nodes()))
nx.set_node_attributes(G, degree_dict, 'degree')

sorted_degree = sorted(degree_dict.items(), key=itemgetter(1), reverse=True)


print("Top 20 nodes by degree:")
for d in sorted_degree[:20]:
    print(d)

betweenness_dict = nx.betweenness_centrality(G) # Run betweenness centrality
eigenvector_dict = nx.eigenvector_centrality(G) # Run eigenvector centrality

# Assign each to an attribute in your network
nx.set_node_attributes(G, betweenness_dict, 'betweenness')
nx.set_node_attributes(G, eigenvector_dict, 'eigenvector')


sorted_betweenness = sorted(betweenness_dict.items(), key=itemgetter(1), reverse=True)

print("Top 20 nodes by betweenness centrality:")
for b in sorted_betweenness[:20]:
    print(b)

    
#First get the top 20 nodes by betweenness as a list
top_betweenness = sorted_betweenness[:20]

#Then find and print their degree
for tb in top_betweenness: # Loop through top_betweenness
    degree = degree_dict[tb[0]] # Use degree_dict to access a node's degree, see footnote 2
    print("Name:", tb[0], "| Betweenness Centrality:", tb[1], "| Degree:", degree)
    

        
# nx.write_gexf(G, '{}_network.gexf'.format(country))
nx.write_gexf(G, 'allCountries_network.gexf')

Network density: 0.0009163569560384626
False
Network diameter of largest component: 14
Triadic closure: 0.004107573466301549
Top 20 nodes by degree:
('869180784728518656', 445)
('921295392024809475', 233)
('242706019', 176)
('267090518', 166)
('4437233895', 117)
('909028875115667456', 93)
('847132974814543872', 91)
('240324087', 89)
('847791625673822208', 83)
('858455197215059968', 83)
('981925161271250945', 79)
('1259387862', 63)
('4224729994', 62)
('3587272518', 62)
('1265545812', 60)
('1624329463', 60)
('1664006540', 60)
('357128018', 60)
('3364189857', 58)
('704007117695524868', 54)
Top 20 nodes by betweenness centrality:
('869180784728518656', 0.32072022271234174)
('242706019', 0.08915602784494524)
('921295392024809475', 0.087567733677026)
('2249280120', 0.0860198583349315)
('267090518', 0.08518123502094518)
('1219308397', 0.07734748814967439)
('4224729994', 0.06792811459711051)
('4437233895', 0.058640656768999905)
('1+ONm2+Ue7XxSmnKUSQHu0Y0x7sg23Y0YY7Q5wlUJOk=', 0.057263629608486

In [ ]:
from networkx.algorithms import community
communities_generator = community.girvan_newman(G)
top_level_communities = next(communities_generator)
next_level_communities = next(communities_generator)
sorted(map(sorted, next_level_communities))

In [ ]:
communities = community.greedy_modularity_communities(G)

modularity_dict = {} # Create a blank dictionary
for i,c in enumerate(communities): # Loop through the list of communities, keeping track of the number for the community
    for name in c: # Loop through each person in a community
        modularity_dict[name] = i # Create an entry in the dictionary for the person, where the value is which group they belong to.

# Now you can add modularity information like we did the other metrics
nx.set_node_attributes(G, modularity_dict, 'modularity')


# First get a list of just the nodes in that class
class0 = [n for n in G.nodes() if G.nodes[n]['modularity'] == 0]

# Then create a dictionary of the eigenvector centralities of those nodes
class0_eigenvector = {n:G.nodes[n]['eigenvector'] for n in class0}

# Then sort that dictionary and print the first 5 results
class0_sorted_by_eigenvector = sorted(class0_eigenvector.items(), key=itemgetter(1), reverse=True)

print("Modularity Class 0 Sorted by Eigenvector Centrality:")
for node in class0_sorted_by_eigenvector[:5]:
    print("Name:", node[0], "| Eigenvector Centrality:", node[1])
    
for i,c in enumerate(communities): # Loop through the list of communities
    if len(c) > 2: # Filter out modularity classes with 2 or fewer nodes
        print('Class '+str(i)+':', list(c)) # Print out the classes and their members
